# FloodLens: track 084 (Jul 14 to Jul 26), Lahore change layer

Run the cells top to bottom. Raw files stay on Colab's temporary disk (they vanish when the session resets). The zip at the end holds only the small outputs.

**Untested code.** The download cell is the most likely to need a fix. If it can't find the scenes, use the manual-URL cell right after it.

In [ ]:
!pip -q install rasterio asf_search

In [ ]:
import os, glob, getpass

RAW = "/content/raw"
OUT = "/content/out"
os.makedirs(RAW, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

BEFORE_NAME = "NISAR_L2_PR_GCOV_025_084_A_017_4005_DHDH_A_20260714T002131_20260714T002205_P05023_F_F_J_001"
AFTER_NAME  = "NISAR_L2_PR_GCOV_026_084_A_017_4005_DHDH_A_20260726T002130_20260726T002205_P05023_N_F_J_001"

# Lahore box taken from the track 135 grid (EPSG:32643, pixel-centre coordinates).
# Columns 13867-17684 and rows 6449-9356 on a grid that starts at x=275765, y=3567595, 10 m pixels.
BOX_UTM43 = dict(
    xmin=275765 + 13867 * 10, xmax=275765 + 17684 * 10,
    ymin=3567595 - 9356 * 10, ymax=3567595 - 6449 * 10,
)

WATER_DB = -15.0    # "dark enough to be water" (check the percentiles printed later)
DROP_DB = -3.0      # change threshold after removing the scene-wide offset
WIN = 7             # smoothing window (7 x 10 m = 70 m)
MIN_PIXELS = 50     # drop blobs under 0.5 ha

In [ ]:
import asf_search as asf

user = input("Earthdata username: ")
pw = getpass.getpass("Earthdata password: ")
session = asf.ASFSession().auth_with_creds(user, pw)

def find_granule(name):
    attempts = (
        lambda: asf.granule_search([name]),
        lambda: asf.search(granule_list=[name], maxResults=5),
    )
    for attempt in attempts:
        try:
            res = attempt()
            if len(res):
                return res[0]
        except Exception as e:
            print("  search attempt failed:", str(e)[:120])
    return None

for name in (BEFORE_NAME, AFTER_NAME):
    path = os.path.join(RAW, name + ".h5")
    if os.path.exists(path):
        print("already here:", name[:45])
        continue
    r = find_granule(name)
    if r is None:
        print("NOT FOUND by search:", name[:45], "-> use the manual-URL cell below")
        continue
    print("downloading", name[:45], "...")
    r.download(path=RAW, session=session)
print("done")

**Only if the search cell could not find a scene:** in Vertex, find the scene, right-click the download icon on the "Science Data" row, choose *Copy link address*, and paste it below. Then run this cell.

In [ ]:
MANUAL_URLS = {}   # e.g. {BEFORE_NAME: "https://...h5", AFTER_NAME: "https://...h5"}

for name, url in MANUAL_URLS.items():
    dest = os.path.join(RAW, name + ".h5")
    if os.path.exists(dest):
        continue
    print("downloading", name[:45])
    with session.get(url, stream=True) as r:
        r.raise_for_status()
        with open(dest, "wb") as fh:
            for chunk in r.iter_content(1 << 20):
                fh.write(chunk)
print("manual step finished")

In [ ]:
def pick(prefix):
    hits = glob.glob(os.path.join(RAW, prefix + "*.h5"))
    if len(hits) != 1:
        raise RuntimeError(f"Expected 1 file starting with {prefix} in {RAW}, found {len(hits)}")
    return hits[0]

before_path = pick("NISAR_L2_PR_GCOV_025_084")   # Jul 14
after_path = pick("NISAR_L2_PR_GCOV_026_084")    # Jul 26

for p in (before_path, after_path):
    gb_size = os.path.getsize(p) / 1e9
    print(f"{os.path.basename(p)[:50]}...  {gb_size:.2f} GB")
    if gb_size < 1:
        raise RuntimeError("File looks too small: the download probably broke. Delete it and download again.")

In [ ]:
import h5py
import numpy as np

GRID = "/science/LSAR/GCOV/grids/frequencyA"

def read_grid(path):
    with h5py.File(path, "r") as f:
        g = f[GRID]
        out = dict(
            x=g["xCoordinates"][:], y=g["yCoordinates"][:],
            epsg=int(g["projection"][()]), shape=g["HHHH"].shape,
            pols=[p.decode() for p in g["listOfPolarizations"][:]],
        )
        try:
            ident = f["/science/LSAR/identification"]
            for k in ("trackNumber", "frameNumber", "orbitPassDirection", "zeroDopplerStartTime"):
                v = ident[k][()]
                out[k] = v.decode() if isinstance(v, bytes) else v
        except Exception as e:
            out["ident_error"] = str(e)[:80]
    return out

gb, ga = read_grid(before_path), read_grid(after_path)
for name, g in (("Jul 14", gb), ("Jul 26", ga)):
    print(name, {k: g[k] for k in g if k not in ("x", "y")})
    print("   x", g["x"][0], "to", g["x"][-1], "| y", g["y"][0], "to", g["y"][-1],
          "| spacing", g["x"][1] - g["x"][0], g["y"][1] - g["y"][0])

same = (gb["epsg"] == ga["epsg"] and gb["shape"] == ga["shape"]
        and np.array_equal(gb["x"], ga["x"]) and np.array_equal(gb["y"], ga["y"]))
print("\nSame grid:", same)
if not same:
    raise RuntimeError("The two scans do not share a grid. Stop and send me this output.")

In [ ]:
from rasterio.warp import transform as warp_transform

b = BOX_UTM43
cx = [b["xmin"], b["xmax"], b["xmin"], b["xmax"]]
cy = [b["ymin"], b["ymin"], b["ymax"], b["ymax"]]
lon, lat = warp_transform("EPSG:32643", "EPSG:4326", cx, cy)
print("Lahore box: lon %.3f to %.3f, lat %.3f to %.3f" % (min(lon), max(lon), min(lat), max(lat)))

fx, fy = warp_transform("EPSG:4326", f"EPSG:{gb['epsg']}", lon, lat)
x, y = gb["x"], gb["y"]
c0 = int(np.searchsorted(x, min(fx)))
c1 = int(np.searchsorted(x, max(fx)))
r0 = int(np.searchsorted(-y, -max(fy)))
r1 = int(np.searchsorted(-y, -min(fy)))
print(f"Window: rows {r0}-{r1}, cols {c0}-{c1}  ({r1 - r0} x {c1 - c0} pixels) of a {len(y)} x {len(x)} grid")

if c1 - c0 < 100 or r1 - r0 < 100:
    raise RuntimeError("The Lahore box is outside this frame. Track 084 frame 017 does not cover it.")
if c0 <= 0 or r0 <= 0 or c1 >= len(x) or r1 >= len(y):
    print("WARNING: the box touches the edge of the frame, so part of it may be empty.")

In [ ]:
import rasterio
from rasterio.transform import from_origin
from rasterio.features import sieve

dx = float(x[1] - x[0])
dy = float(y[0] - y[1])
transform = from_origin(float(x[c0]) - dx / 2, float(y[r0]) + dy / 2, dx, dy)
crs = f"EPSG:{gb['epsg']}"

def read_db(path):
    with h5py.File(path, "r") as f:
        lin = f[GRID]["HHHH"][r0:r1, c0:c1].astype("float32")
    ok = np.isfinite(lin) & (lin > 0)
    db = np.full(lin.shape, np.nan, dtype="float32")
    db[ok] = 10.0 * np.log10(lin[ok])
    return db

def save_tif(name, arr, dtype="float32", nodata=np.nan):
    with rasterio.open(os.path.join(OUT, name), "w", driver="GTiff",
                       height=arr.shape[0], width=arr.shape[1], count=1, dtype=dtype,
                       crs=crs, transform=transform, nodata=nodata, compress="deflate") as dst:
        dst.write(arr.astype(dtype), 1)

before = read_db(before_path)
after = read_db(after_path)
for n, arr in (("Jul 14", before), ("Jul 26", after)):
    frac = float(np.isfinite(arr).mean())
    print(f"{n}: valid {frac * 100:.1f}%, median {np.nanmedian(arr):.2f} dB")
    if frac < 0.95:
        print("  WARNING: under 95% valid. Part of the box is empty in this scan.")

save_tif("t084_hh_db_20260714.tif", before)
save_tif("t084_hh_db_20260726.tif", after)
print("saved clipped dB GeoTIFFs")

In [ ]:
import matplotlib.pyplot as plt

def box_filter(arr, win):
    pad = win // 2
    p = np.pad(arr, pad, mode="edge").astype("float64")
    c = p.cumsum(axis=0).cumsum(axis=1)
    c = np.pad(c, ((1, 0), (1, 0)))
    s = c[win:, win:] - c[:-win, win:] - c[win:, :-win] + c[:-win, :-win]
    return (s / (win * win)).astype("float32")

def smooth_db(db, win=WIN):
    ok = np.isfinite(db)
    with np.errstate(all="ignore"):
        lin = np.where(ok, 10.0 ** (db / 10.0), 0.0).astype("float32")
    w = box_filter(ok.astype("float32"), win)
    m = box_filter(lin, win)
    out = np.full(db.shape, np.nan, dtype="float32")
    good = w > 0.5
    out[good] = 10.0 * np.log10(m[good] / w[good])
    return out

sb = smooth_db(before)   # Jul 14
sa = smooth_db(after)    # Jul 26
raw_change = sa - sb
offset = float(np.nanmedian(raw_change))
change = raw_change - offset
print(f"Median offset removed: {offset:.2f} dB")
print("Adjusted change percentiles 1/5/50/95/99:", np.round(np.nanpercentile(change, [1, 5, 50, 95, 99]), 2))
print("Smoothed Jul 26 dB percentiles 1/2/5/10:", np.round(np.nanpercentile(sa, [1, 2, 5, 10]), 1))
save_tif("t084_change_smooth_adj_db.tif", change)

step = 4
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(sb[::step, ::step], cmap="gray", vmin=-25, vmax=5); ax[0].set_title("Jul 14 (dB, smoothed)")
ax[1].imshow(sa[::step, ::step], cmap="gray", vmin=-25, vmax=5); ax[1].set_title("Jul 26 (dB, smoothed)")
im = ax[2].imshow(change[::step, ::step], cmap="RdBu", vmin=-8, vmax=8)
ax[2].set_title("Change (Jul 26 - Jul 14), offset removed\nblue = brighter, red = darker")
fig.colorbar(im, ax=ax[2], shrink=0.7)
for a_ in ax:
    a_.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUT, "t084_preview.png"), dpi=110)
plt.show()

In [ ]:
def maxpool(mask, step):
    h = (mask.shape[0] // step) * step
    w = (mask.shape[1] // step) * step
    return mask[:h, :w].reshape(h // step, step, w // step, step).max(axis=(1, 3))

fin = np.isfinite(change)
with np.errstate(all="ignore"):
    cand = fin & (change < DROP_DB) & (sa < WATER_DB) & (sb > WATER_DB)
clean = sieve(cand.astype("uint8"), size=MIN_PIXELS, connectivity=8).astype(bool)
print(f"Candidates: raw {cand.sum() / 1e4:.2f} km2, after removing specks {clean.sum() / 1e4:.2f} km2")
save_tif("t084_candidates.tif", clean, dtype="uint8", nodata=0)

# River corridor from the large water body that is dark in both scans
with np.errstate(all="ignore"):
    both = (sa < WATER_DB) & (sb < WATER_DB)
big = sieve(both.astype("uint8"), size=5000, connectivity=8).astype(bool)
corridor = box_filter(big.astype("float32"), 201) > 0
print(f"Main river body (dark in both): {big.sum() / 1e4:.2f} km2 | corridor: {corridor.sum() / 1e4:.1f} km2\n")

print("Water inside the corridor (km2). 'gained' = dark on Jul 26 but not on Jul 14:")
for t in (-13, -14, -15, -16):
    with np.errstate(all="ignore"):
        wj, wa = sb < t, sa < t
    print(f"  < {t} dB | Jul 14: {(wj & corridor).sum() / 1e4:6.2f} | Jul 26: {(wa & corridor).sum() / 1e4:6.2f} | "
          f"gained: {(wa & ~wj & corridor).sum() / 1e4:5.2f} | lost: {(wj & ~wa & corridor).sum() / 1e4:5.2f}")

print(f"\nCandidates in river corridor: {(clean & corridor).sum() / 1e4:.2f} km2")
print(f"Candidates outside (farmland etc.): {(clean & ~corridor).sum() / 1e4:.2f} km2")

if corridor.any():
    rows = np.where(corridor.any(axis=1))[0]
    cols = np.where(corridor.any(axis=0))[0]
    i0, i1, j0, j1 = rows[0], rows[-1] + 1, cols[0], cols[-1] + 1
    step = 2
    with np.errstate(all="ignore"):
        wj = (sb < WATER_DB)[i0:i1, j0:j1]
        wa = (sa < WATER_DB)[i0:i1, j0:j1]
    base = np.nan_to_num(sa[i0:i1:step, j0:j1:step], nan=-25.0)
    rgb = np.dstack([np.clip((base + 25) / 30, 0, 1)] * 3)
    for mask, color in ((wj & wa, [0.2, 0.4, 1.0]), (wa & ~wj, [1.0, 0.1, 0.1]), (wj & ~wa, [1.0, 0.9, 0.1])):
        m = maxpool(mask, step)
        rgb[:m.shape[0], :m.shape[1]][m] = color
    plt.figure(figsize=(13, 10))
    plt.imshow(rgb)
    plt.title("Ravi corridor: blue = water both dates, red = gained by Jul 26, yellow = lost")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, "t084_ravi_corridor.png"), dpi=110)
    plt.show()
else:
    print("No river body found at this water threshold. Check the Jul 26 percentiles printed above.")

In [ ]:
import shutil
shutil.make_archive("/content/track084_outputs", "zip", OUT)
from google.colab import files
files.download("/content/track084_outputs.zip")

**Send back:** the text printed by the cells (especially the grid check, window, percentiles and corridor table) and the two PNGs.